# 01 — Coleta de dados do IBGE (Censo 2022)

**Objetivo desta etapa:** obter, para os 645 municípios de São Paulo, o número de
idosos (60+) que moram sozinhos e a população idosa total, a partir do Censo
Demográfico 2022.

**Por que esse dado importa para o estudo:** ele é a variável independente da
hipótese — *municípios com mais idosos morando sozinhos têm mais internações
por causas evitáveis?*

**Saída desta etapa:** `data/processed/censo_domicilios_sp.csv` e
`data/processed/municipios_sp.csv` (tabela de códigos de município, que vamos
reusar em todos os notebooks seguintes para cruzar IBGE com DATASUS).

**Duas formas de conseguir o dado, nessa ordem:**
1. **Automática** — via API SIDRA (célula abaixo). Tentamos primeiro porque é
   reprodutível (qualquer pessoa roda o notebook de novo e obtém o mesmo dado).
2. **Manual** — se a tabela/variável certa não estiver disponível pela API,
   baixamos pelo site do SIDRA e salvamos em `data/external/`.

⚠️ **Ponto em aberto que precisamos validar juntas na primeira execução:** o
Censo 2022 tem uma tabela específica de *arranjo domiciliar* (unipessoal) por
faixa etária — mas o número exato dessa tabela no SIDRA muda conforme a
divulgação. Deixei abaixo o caminho manual (mais confiável agora) e o
automático com um número de tabela a confirmar. **Rode as duas células, veja
qual funciona, e me manda o que aparecer (erro ou resultado) que a gente
ajusta juntas.**


In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("..") / "src"))
import config  # noqa: E402

import pandas as pd
import requests


## 1.1 Lista de municípios de SP (código IBGE de 7 dígitos + nome)

Vamos usar a API de localidades do IBGE (não é o Censo em si, é só a lista de
municípios — essa API é estável e não muda). Isso nos dá a tabela de
referência `municipios_sp.csv` que vamos usar em TODOS os notebooks seguintes
para juntar (merge) IBGE com DATASUS pelo código do município.


In [ ]:
url_municipios = f"https://servicodados.ibge.gov.br/api/v1/localidades/estados/{config.UF_SIGLA}/municipios"

resp = requests.get(url_municipios, timeout=30)
resp.raise_for_status()

municipios = pd.DataFrame(resp.json())[["id", "nome"]]
municipios = municipios.rename(columns={"id": "codigo_ibge", "nome": "municipio"})
municipios["codigo_datasus"] = municipios["codigo_ibge"].apply(config.codigo_ibge_para_datasus)

print(f"{len(municipios)} municípios encontrados (esperado: 645)")
municipios.to_csv(config.DATA_PROCESSED / "municipios_sp.csv", index=False)
municipios.head()


## 1.2 Domicílios / arranjo domiciliar — tentativa automática (API SIDRA)

A API do SIDRA tem o formato:

```
https://apisidra.ibge.gov.br/values/t/{tabela}/n6/all/v/{variavel}/p/{periodo}
```

- `t` = número da tabela
- `n6` = nível geográfico "município" (todos os municípios do Brasil — depois
  filtramos para SP)
- `v` = código da variável dentro da tabela
- `p` = período (ano)

Comece rodando a célula abaixo com a tabela 9605 (Domicílios particulares
permanentes, por município — já confirmada em conversa anterior sobre este
projeto). Se o resultado não tiver o recorte por idade/arranjo unipessoal que
precisamos, vá para a seção 1.3 (caminho manual) — é o caminho mais seguro
agora.


In [ ]:
TABELA_SIDRA = 9605       # Domicílios particulares permanentes (ajustar se acharmos tabela mais específica)
VARIAVEL_SIDRA = "all"    # começamos pedindo todas as variáveis disponíveis, para inspecionar
PERIODO_SIDRA = "last"    # último período disponível (Censo 2022)

url_sidra = (
    f"https://apisidra.ibge.gov.br/values/t/{TABELA_SIDRA}"
    f"/n6/all/v/{VARIAVEL_SIDRA}/p/{PERIODO_SIDRA}"
)

try:
    resp = requests.get(url_sidra, timeout=60)
    resp.raise_for_status()
    dados_sidra = resp.json()
    df_sidra = pd.DataFrame(dados_sidra[1:], columns=dados_sidra[0])
    print("Colunas retornadas:", list(df_sidra.columns))
    print(f"{len(df_sidra)} linhas")
except Exception as e:
    print("A chamada automática falhou ou a tabela não é a certa.")
    print("Erro:", e)
    print("\n>>> Siga a seção 1.3 (download manual) e me avise o que apareceu aqui.")
    df_sidra = None

df_sidra.head() if df_sidra is not None else None


Se a célula acima trouxe colunas como `"Município"`, `"Valor"` e alguma
classificação de idade/tipo de domicílio: ótimo, copie aqui o nome exato das
colunas que representam **idade do responsável** e **tipo de arranjo
(unipessoal)** — eu ajusto a célula seguinte para filtrar certo.

Se deu erro, ou a tabela não tem esse recorte: siga a seção 1.3.


## 1.3 Caminho manual (recomendado por enquanto)

1. Acesse: https://sidra.ibge.gov.br/tabela/9605
2. Nível territorial: **Município** → filtrar Unidade da Federação = **São Paulo**
3. Se existir a opção de classificar por **grupo de idade da pessoa
   responsável** e **tipo de arranjo domiciliar** (unipessoal), selecione
   idade 60 ou mais e arranjo unipessoal.
4. Formato de download: **CSV**
5. Salve o arquivo como `censo_domicilios_sp.csv` dentro de `data/external/`

Se essa tabela específica não existir no SIDRA (a divulgação por arranjo
domiciliar do Censo 2022 é recente e o número pode mudar), procure no SIDRA
por **"arranjo domiciliar"** ou **"domicílio unipessoal"**, filtre por
Município = SP, e me manda o número da tabela que você achou — eu adapto a
célula 1.2 para usar ela automaticamente da próxima vez.


In [ ]:
caminho_manual = config.DATA_EXTERNAL / "censo_domicilios_sp.csv"

if caminho_manual.exists():
    # tente ; e , como separador, e latin1/utf-8 como encoding -- exportações do SIDRA variam
    for sep in (";", ","):
        for encoding in ("utf-8", "latin1"):
            try:
                df_censo = pd.read_csv(caminho_manual, sep=sep, encoding=encoding)
                if df_censo.shape[1] > 1:
                    print(f"Lido com sep='{sep}' encoding='{encoding}'")
                    break
            except Exception:
                continue
        else:
            continue
        break
    print(df_censo.shape)
    df_censo.head()
else:
    print(f"Ainda não encontrei o arquivo em {caminho_manual}")
    print("Baixe conforme a seção 1.3 e rode esta célula de novo.")
    df_censo = None


## 1.4 Consolidar e salvar

Depois de validar (automático ou manual) que temos, por município de SP:
- total de idosos (60+)
- idosos (60+) morando sozinhos (arranjo unipessoal)

ajuste a célula abaixo para selecionar/renomear essas colunas e salvar o
resultado final. Deixei o esqueleto pronto — **essa é a célula que vamos
fechar juntas na primeira rodada**, porque depende de qual caminho (1.2 ou
1.3) funcionou e de como as colunas vieram nomeadas.


In [ ]:
# ESQUELETO — ajustar nomes de coluna conforme o que veio na seção 1.2 ou 1.3
#
# df_final = df_censo.rename(columns={
#     "Município (Código)": "codigo_ibge",
#     "Valor": "idosos_sozinhos",   # ou o nome real da coluna de valor
# })[["codigo_ibge", "idosos_sozinhos"]]
#
# df_final["codigo_ibge"] = df_final["codigo_ibge"].astype(int)
# df_final.to_csv(config.DATA_PROCESSED / "censo_domicilios_sp.csv", index=False)
# df_final.head()

print("Ajuste esta célula com base no resultado da 1.2/1.3 e me chama para revisarmos juntas.")
